In [ ]:
import pandas as pd
import numpy as np
import altair

# Crear els Dataframes del XML

## Location

In [ ]:
location_columns = ["numeroparte", "idcomunidad", "idprovincia",
                    "idmunicipio", "nummunicipiosafectados", "latitud", "longitud"]

locations = pd.read_xml("dades/Xml_20261116_123725_1.xml", xpath="./Pif/pif_localizacion")[location_columns]
locations.head()

In [ ]:
municipis_dades = pd.read_excel("dades/diccionario26.xlsx", header=1)
municipis_dades = municipis_dades.loc[municipis_dades["CODAUTO"]==9, ["CPRO", "CMUN", "DC", "NOMBRE"]]
print(municipis_dades.loc[municipis_dades["CPRO"]==8])
municipis_dades.head()

In [ ]:
provincies = ["08", "17", "25", "43"]

path = "dades/MUNICIPIOS.csv"

municipis_coordenades = pd.read_csv(
    path,
    encoding="cp1252",
    sep=";",
    dtype="string",
)

municipis_coordenades = municipis_coordenades.loc[municipis_coordenades["COD_PROV"].isin(provincies)]
municipis_coordenades.info()
municipis_coordenades.head()

In [ ]:
def municipality_code(province, municipality):
    return (
        province.astype("Int64").astype("string").str.zfill(2)
        + municipality.astype("Int64").astype("string").str.zfill(3)
    )


def parse_coordinate(values):
    text = values.astype("string").str.strip().str.replace(",", ".", regex=False)
    return pd.to_numeric(text, errors="coerce")


fires = locations.assign(
    numero_parte=locations["numeroparte"],
    codi_municipi=municipality_code(locations["idprovincia"], locations["idmunicipio"]),
    latitud=parse_coordinate(locations["latitud"]),
    longitud=parse_coordinate(locations["longitud"]),
)

towns = municipis_dades.assign(
    codi_municipi=municipality_code(municipis_dades["CPRO"], municipis_dades["CMUN"]),
    municipi=municipis_dades["NOMBRE"].str.strip(),
)[["codi_municipi", "municipi"]]

coords = municipis_coordenades.assign(
    codi_municipi=municipis_coordenades["COD_INE"].str.strip().str[:5],
    latitud_municipi=parse_coordinate(municipis_coordenades["LATITUD_ETRS89_REGCAN95"]),
    longitud_municipi=parse_coordinate(municipis_coordenades["LONGITUD_ETRS89_REGCAN95"]),
)[["codi_municipi", "latitud_municipi", "longitud_municipi"]]

provinces = (
    municipis_coordenades[["COD_PROV", "PROVINCIA"]]
    .rename(columns={"COD_PROV": "idprovincia", "PROVINCIA": "provincia"})
    .assign(
        idprovincia=lambda df: df["idprovincia"].astype("Int64"),
        provincia=lambda df: df["provincia"].str.strip(),
    )
    .dropna(subset=["idprovincia"])
    .drop_duplicates()
)

merged = (
    fires.merge(towns.dropna(subset=["codi_municipi"]), on="codi_municipi",
                how="left", validate="many_to_one")
    .merge(coords.dropna(subset=["codi_municipi"]), on="codi_municipi",
           how="left", validate="many_to_one")
    .merge(provinces, on="idprovincia", how="left", validate="many_to_one")
)

merged["latitud"] = merged["latitud"].fillna(merged["latitud_municipi"])
merged["longitud"] = merged["longitud"].fillna(merged["longitud_municipi"])

locations_enriched = merged[[
    "numero_parte",
    "idprovincia",
    "idmunicipio",
    "municipi",
    "provincia",
    "latitud",
    "longitud",
    "nummunicipiosafectados",
]]

In [ ]:
comarques = pd.read_csv(
    "dades/corrMpisComsAtps.csv",
    sep=";",
    skiprows=3,
    usecols=["Codi municipi", "Nom comarca"],
    dtype={"Codi municipi": "string"},
)[["Codi municipi", "Nom comarca"]].copy()

comarques["codi_municipi"] = comarques["Codi municipi"].str.strip().str[:5]

comarca_by_municipality = comarques.set_index(
    "codi_municipi", verify_integrity=True
)["Nom comarca"]

locations_enriched = locations_enriched.assign(
    comarca=merged["codi_municipi"].map(comarca_by_municipality)
)


In [ ]:
locations_enriched.info()
locations.info()

In [ ]:
locations_enriched.loc[locations_enriched["municipi"].notna()].head()

## Causes

In [ ]:
causes = pd.read_xml("dades/Xml_20261116_123725_1.xml", xpath="./Pif/pif_causa")
causes.info()
causes.head()

In [ ]:
columnes_causes = ["numeroparte", "idcausa", "causa"]

causes_egif_ca = pd.read_csv("dades/causes_egif_ca.csv")
causes_enriched = causes.merge(
    causes_egif_ca,
    on="idcausa",
    how="left",
    validate="many_to_one",
)
causes_enriched = causes_enriched[columnes_causes]
causes_enriched.head()

# Bla bla bla

Estic processant la nova base de dades